In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('../outputs/finance.db')
print("Connecté à finance.db")

query = """
SELECT 
    dt.type,
    COUNT(*)                          AS nb_transactions,
    ROUND(SUM(ft.amount), 2)          AS montant_total,
    ROUND(AVG(ft.amount), 2)          AS montant_moyen,
    ROUND(MAX(ft.amount), 2)          AS montant_max
FROM fact_transactions ft
JOIN dim_type_transaction dt ON ft.type_id = dt.type_id
GROUP BY dt.type
ORDER BY montant_total DESC
"""
df = pd.read_sql_query(query, conn)
print(df)

query = """
SELECT 
    dt.type,
    COUNT(*)                                        AS nb_transactions,
    SUM(ft.est_fraude)                              AS nb_fraudes,
    ROUND(SUM(ft.est_fraude) * 100.0 / COUNT(*), 2) AS taux_fraude_pct
FROM fact_transactions ft
JOIN dim_type_transaction dt ON ft.type_id = dt.type_id
GROUP BY dt.type
ORDER BY taux_fraude_pct DESC
"""
df = pd.read_sql_query(query, conn)
print(df)


query = """
SELECT 
    dd.departement,
    COUNT(DISTINCT fe.employe_id)   AS nb_employes,
    ROUND(AVG(fe.age), 1)           AS age_moyen,
    ROUND(AVG(fe.anciennete), 1)    AS anciennete_moyenne
FROM fact_employes fe
JOIN dim_departement dd ON fe.dept_id = dd.dept_id
WHERE fe.annee = (SELECT MAX(annee) FROM fact_employes)
GROUP BY dd.departement
ORDER BY nb_employes DESC
"""
df = pd.read_sql_query(query, conn)
print(df)


query = """
SELECT 
    fe.raison_depart,
    COUNT(*)                                     AS nb_departs,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS pct_total
FROM fact_employes fe
WHERE fe.statut = 'TERMINATED'
GROUP BY fe.raison_depart
ORDER BY nb_departs DESC
"""
df = pd.read_sql_query(query, conn)
print(df)

query = """
SELECT 
    COUNT(*) AS nb_incoherences
FROM fact_transactions
WHERE ROUND(solde_avant_source - amount, 2) != ROUND(solde_apres_source, 2)
AND solde_apres_source != 0
"""
df = pd.read_sql_query(query, conn)
print(f"Transactions avec soldes incohérents : {df['nb_incoherences'].values[0]}")


# On resauvegarde les résultats clés dans outputs/
q1 = pd.read_sql_query("SELECT dt.type, COUNT(*) AS nb, ROUND(SUM(ft.amount),2) AS total FROM fact_transactions ft JOIN dim_type_transaction dt ON ft.type_id=dt.type_id GROUP BY dt.type", conn)
q1.to_csv('../outputs/analyse_transactions.csv', index=False)

q2 = pd.read_sql_query("SELECT dd.departement, COUNT(DISTINCT fe.employe_id) AS nb_employes FROM fact_employes fe JOIN dim_departement dd ON fe.dept_id=dd.dept_id GROUP BY dd.departement", conn)
q2.to_csv('../outputs/analyse_departements.csv', index=False)

conn.close()
print("Résultats sauvegardés dans outputs/")

Connecté à finance.db
       type  nb_transactions  montant_total  montant_moyen  montant_max
0  TRANSFER           532909   4.852920e+11      910647.01  92445516.64
1  CASH_OUT          2237484   3.944130e+11      176275.22  10000000.00
2   CASH_IN          1399284   2.363674e+11      168920.24   1915267.90
3   PAYMENT          2151495   2.809337e+10       13057.60    238637.98
4     DEBIT            41432   2.271992e+08        5483.67    569077.51
       type  nb_transactions  nb_fraudes  taux_fraude_pct
0  TRANSFER           532909        4097             0.77
1  CASH_OUT          2237484        4100             0.18
2   PAYMENT          2151495           0             0.00
3     DEBIT            41432           0             0.00
4   CASH_IN          1399284           0             0.00
             departement  nb_employes  age_moyen  anciennete_moyenne
0       Customer Service          966       31.9                 7.6
1                  Meats          911       55.8            